# RDST + sensor spread + ridge (seed 0)

One model, the same for process A and process B:

- every batch becomes 1000 points per sensor (slice means);
- **RDST** (random dilated shapelets) picks 10,000 short patterns from the training batches and measures how close
  every batch gets to each of them;
- slice means wipe out fast noise, so a small **spread** block adds, per sensor, how much it moves inside the batch
  (std, mean step size, range);
- one **ridge** classifier decides. Its alpha and the weight of the spread block are picked by 4-fold CV on F1 of
  "rejected", inside the training data only.

1. Check the model on the train/test split from `train_test_split_process_A.ipynb` (A and B).
2. Train on all calibration data (A and B) and write a table in the leaderboard format.

Same model as `rdst1000_spread_f1w` in `src/night.py`. Needs `aeon`. The data folder is only read, never written.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from aeon.transformations.collection.shapelet_based import RandomDilatedShapeletTransform
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
LENGTH = 1000  # every batch becomes 1000 points per sensor
ALPHAS = np.logspace(-3, 5, 9)  # ridge strengths to try
SPREAD_WEIGHTS = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]  # weight of the spread block next to RDST

# first data folder that exists (Renku or a local copy)
candidates = ["../../input-data/Project 2 - BRPMM", "../input-data/Project 2 - BRPMM",
              "/home/renku/work/input-data/Project 2 - BRPMM", "../../Project 2 - BRPMM"]
DATA_DIR = next(Path(c) for c in candidates if Path(c).is_dir())
print(DATA_DIR.resolve())

In [ ]:
def load_batches(csv_path):
    """One array (rows x sensors) per batch."""
    df = pd.read_csv(csv_path)
    sensors = [c for c in df.columns if c != "batch_id"]
    return {b: g[sensors].to_numpy(float) for b, g in df.groupby("batch_id")}


def fixed_length(a):
    """Cut the batch into LENGTH equal slices and average each one -> (sensors x LENGTH)."""
    n = len(a)
    if n < LENGTH:  # short batch: stretch it instead
        old, new = np.linspace(0, 1, n), np.linspace(0, 1, LENGTH)
        return np.stack([np.interp(new, old, a[:, j]) for j in range(a.shape[1])])
    slice_id = np.arange(n) * LENGTH // n
    sums = np.zeros((LENGTH, a.shape[1]))
    np.add.at(sums, slice_id, a)
    return (sums / np.bincount(slice_id, minlength=LENGTH)[:, None]).T


def spread(a):
    """How much each sensor moves inside the batch: log of std, of mean step size and of range."""
    moves = np.concatenate([a.std(0), np.abs(np.diff(a, axis=0)).mean(0), a.max(0) - a.min(0)])
    return np.log(moves + 1e-6)


def inputs(batches, ids):
    series = np.stack([fixed_length(batches[b]) for b in ids])  # (batches, sensors, LENGTH)
    return series, np.stack([spread(batches[b]) for b in ids])


def mean_f1(X, y, alpha):
    """F1 of "rejected" over a 4-fold CV inside the training data."""
    scores = []
    for a, b in StratifiedKFold(4, shuffle=True, random_state=SEED).split(X, y):
        pred = RidgeClassifier(alpha=alpha, class_weight="balanced").fit(X[a], y[a]).predict(X[b])
        scores.append(f1_score(y[b], pred, zero_division=0))
    return np.mean(scores)


class Model:
    def fit(self, series, spread_block, y):
        # put every sensor on the same scale (training statistics only)
        self.mean = series.mean(axis=(0, 2), keepdims=True)
        self.std = series.std(axis=(0, 2), keepdims=True)
        self.std[self.std == 0] = 1.0
        self.rdst = RandomDilatedShapeletTransform(random_state=SEED, n_jobs=1)
        shapes = self._clean(self.rdst.fit_transform(self._scale(series), y))
        self.scalers = [StandardScaler().fit(shapes), StandardScaler().fit(spread_block)]

        # pick the spread weight and alpha with the best CV F1 (first best on ties)
        best = -1
        for w in SPREAD_WEIGHTS:
            X = self._join(shapes, spread_block, w)
            for alpha in ALPHAS:
                score = mean_f1(X, y, alpha)
                if score > best + 1e-12:
                    best, self.weight, self.alpha = score, w, alpha
        print(f"spread weight {self.weight}, alpha {self.alpha:g}, CV F1 inside training {best:.3f}")
        X = self._join(shapes, spread_block, self.weight)
        self.ridge = RidgeClassifier(alpha=self.alpha, class_weight="balanced").fit(X, y)
        return self

    def predict(self, series, spread_block):
        shapes = self._clean(self.rdst.transform(self._scale(series)))
        return self.ridge.predict(self._join(shapes, spread_block, self.weight))  # 1 = rejected

    def _scale(self, series):
        return (series - self.mean) / self.std

    def _clean(self, F):
        return np.nan_to_num(np.asarray(F, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    def _join(self, shapes, spread_block, w):
        """Standardise both blocks; each block gets a total weight (1 for RDST, w for spread),
        so 30,000 RDST numbers do not drown the 3 numbers per sensor of the spread block."""
        a = self.scalers[0].transform(shapes) / np.sqrt(shapes.shape[1])
        b = self.scalers[1].transform(spread_block) * w / np.sqrt(spread_block.shape[1])
        return np.hstack([a, b])

## 1. Check on the colleague's split

Same split as `train_test_split_process_A.ipynb`: 30% of the batches for testing, stratified by label,
`random_state=42`. The same split is used for process B (it has only 5 rejected batches, so 1 goes to the test part).

In [ ]:
for p in ["A", "B"]:
    folder = DATA_DIR / f"process {p}" / "calibration data"
    batches = load_batches(folder / "calibration_data.csv")
    labels = pd.read_csv(folder / "calibration_labels.csv")
    is_rejected = labels.set_index("batch_id").label.eq("rejected").astype(int)

    train_ids, test_ids = train_test_split(labels.batch_id, test_size=0.30, random_state=42, stratify=labels.label)
    train_ids, test_ids = train_ids.sort_index(), test_ids.sort_index()  # keep the file order

    model = Model().fit(*inputs(batches, train_ids), is_rejected[train_ids].to_numpy())
    pred = model.predict(*inputs(batches, test_ids))
    true = is_rejected[test_ids].to_numpy()

    print(f"process {p}: F1 rejected {f1_score(true, pred):.3f} | precision {precision_score(true, pred):.3f} | "
          f"recall {recall_score(true, pred):.3f} | accuracy {accuracy_score(true, pred):.3f}")
    print(f"  rejected caught: {((pred == 1) & (true == 1)).sum()} of {true.sum()}, "
          f"false alarms: {((pred == 1) & (true == 0)).sum()} of {(true == 0).sum()}\n")

## 2. Train on all calibration data and predict the test data

One model per process, same architecture. The table goes to `outputs/` (not sent anywhere).

In [ ]:
predictions = []
for p in ["A", "B"]:
    folder = DATA_DIR / f"process {p}"
    batches = load_batches(folder / "calibration data" / "calibration_data.csv")
    labels = pd.read_csv(folder / "calibration data" / "calibration_labels.csv")
    model = Model().fit(*inputs(batches, labels.batch_id), labels.label.eq("rejected").astype(int).to_numpy())

    test = load_batches(folder / "test data" / "test_data.csv")
    ids = sorted(test)
    pred = model.predict(*inputs(test, ids))
    predictions.append(pd.DataFrame({"batch_id": ids, "label": np.where(pred == 1, "rejected", "accepted")}))
    print(f"process {p}: {pred.sum()} of {len(pred)} test batches predicted rejected")

# same rows and order as the template
template = pd.read_csv(DATA_DIR / "BRPMM_template.csv")
template.columns = template.columns.str.strip()
template["batch_id"] = template.batch_id.str.strip()
submission = template[["batch_id", "project"]].merge(pd.concat(predictions), on="batch_id", how="left")
assert len(submission) == len(template) and submission.label.notna().all()

Path("../outputs").mkdir(exist_ok=True)
submission.to_csv("../outputs/submission_rdst_spread_seed0.csv", index=False)
submission.groupby(["project", "label"]).size()